In [1]:
print(0)

0


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

import glob
import random
import numpy as np
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from timm.utils import ModelEmaV3
from tqdm import tqdm
from torchvision.utils import make_grid
from collections import defaultdict

from pytorch_msssim import ssim

In [3]:
BONE_MARROW_FOLDER_PATH = '/home/pratyushprasad/DPO_PROJECT/0. format_dataset/bone_marrow'
CLASSES = ['PLM', 'PEB', 'MYB', 'NGS', 'BLA', 'OTH', 'NIF', 'EBO', 'PMO', 'HAC', 'ART', 'MMZ', 'NGB', 'MON', 'BAS', 'LYT', 'EOS']

In [4]:
def set_seed(seed: int = 42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)

def get_transforms(image_size=(128, 128)):
    return transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

In [5]:
def load_image_from_path(img_path, transform):
    try:
        img = Image.open(img_path).convert('RGB')
        img_tensor = transform(img)
        return img_tensor
    except Exception as e:
        print(f"Error loading {img_path}: {e}")
        return None

def get_image_folders_for_class(class_folder_path):
    if not os.path.exists(class_folder_path):
        print(f"Warning: {class_folder_path} does not exist!")
        return []

    image_folders = []
    for item in os.listdir(class_folder_path):
        item_path = os.path.join(class_folder_path, item)
        if os.path.isdir(item_path):
            image_folders.append(item_path)
    image_folders.sort(key=lambda x: int(x.split('_')[-1]) if x.split('_')[-1].isdigit() else 0)
    return image_folders

In [6]:
def load_triplet_from_image_folder(image_folder_path, transform):
    if not os.path.exists(image_folder_path):
        return None, None, None

    image_extensions = ['*.jpg', '*.jpeg', '*.png']
    all_files = []
    for ext in image_extensions:
        all_files.extend(glob.glob(os.path.join(image_folder_path, ext)))

    real_image = None
    plausible_images = []
    implausible_images = []

    for file_path in all_files:
        filename = os.path.basename(file_path).lower()
        if filename.startswith('real'):
            real_image = file_path
        elif filename.startswith('plausible'):
            plausible_images.append(file_path)
        elif filename.startswith('implausible'):
            implausible_images.append(file_path)

    real_tensor = None
    plausible_tensor = None
    implausible_tensor = None

    if real_image:
        real_tensor = load_image_from_path(real_image, transform)

    if plausible_images:
        selected_plausible = random.choice(plausible_images)
        plausible_tensor = load_image_from_path(selected_plausible, transform)

    if implausible_images:
        selected_implausible = random.choice(implausible_images)
        implausible_tensor = load_image_from_path(selected_implausible, transform)

    return real_tensor, plausible_tensor, implausible_tensor

In [7]:
def create_triplet_dataset():
    print("Loading images for all classes...")
    all_triplets = []
    transform = get_transforms()

    for class_name in CLASSES:
        print(f"Processing class: {class_name}")

        class_folder_path = os.path.join(BONE_MARROW_FOLDER_PATH, class_name)
        image_folders = get_image_folders_for_class(class_folder_path)

        if len(image_folders) < 50:
            print(f"Warning: Class {class_name} has only {len(image_folders)} image folders, need at least 050")
            continue

        print(f"  Processing first 050 image folders...")
        class_triplets = []

        for i in range(min(50, len(image_folders))):
            image_folder = image_folders[i]
            real, plausible, implausible = load_triplet_from_image_folder(image_folder, transform)

            if real is not None and plausible is not None and implausible is not None:
                class_triplets.append((real, plausible, implausible))
            else:
                print(f"    Warning: Incomplete triplet in {image_folder}")

        print(f"  Loaded {len(class_triplets)} triplets for class {class_name}")
        all_triplets.extend(class_triplets)

    total_triplets = len(all_triplets)
    train_size = int(0.8 * total_triplets)

    train_triplets = all_triplets[:train_size]
    val_triplets = all_triplets[train_size:]

    print(f"Total triplets: {total_triplets}")
    print(f"Train triplets: {len(train_triplets)} (80%)")
    print(f"Validation triplets: {len(val_triplets)} (20%)")

    return train_triplets, val_triplets

In [8]:
class ResBlock(nn.Module):
    def __init__(self, C: int, num_groups: int, dropout_prob: float):
        super().__init__()
        self.relu = nn.ReLU(inplace=True)
        self.gnorm1 = nn.GroupNorm(num_groups=num_groups, num_channels=C)
        self.gnorm2 = nn.GroupNorm(num_groups=num_groups, num_channels=C)
        self.conv1 = nn.Conv2d(C, C, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(C, C, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(p=dropout_prob, inplace=True)

    def forward(self, x, embeddings):
        if embeddings.shape[1] > x.shape[1]:
            embeddings = embeddings[:, :x.shape[1], :, :]
        elif embeddings.shape[1] < x.shape[1]:
            pad_size = x.shape[1] - embeddings.shape[1]
            padding = torch.zeros(embeddings.shape[0], pad_size, embeddings.shape[2], embeddings.shape[3], device=embeddings.device)
            embeddings = torch.cat([embeddings, padding], dim=1)

        x = x + embeddings
        r = self.conv1(self.relu(self.gnorm1(x)))
        r = self.dropout(r)
        r = self.conv2(self.relu(self.gnorm2(r)))
        return r + x

class Attention(nn.Module):
    def __init__(self, C: int, num_heads: int, dropout_prob: float):
        super().__init__()
        self.proj1 = nn.Linear(C, C*3)
        self.proj2 = nn.Linear(C, C)
        self.num_heads = num_heads
        self.dropout_prob = dropout_prob

    def forward(self, x):
        h, w = x.shape[2:]
        x = rearrange(x, 'b c h w -> b (h w) c')
        x = self.proj1(x)
        x = rearrange(x, 'b L (C H K) -> K b H L C', K=3, H=self.num_heads)
        q, k, v = x[0], x[1], x[2]
        x = F.scaled_dot_product_attention(q, k, v, is_causal=False, dropout_p=self.dropout_prob)
        x = rearrange(x, 'b H (h w) C -> b h w (C H)', h=h, w=w)
        x = self.proj2(x)
        return rearrange(x, 'b h w C -> b C h w')

class UnetLayer(nn.Module):
    def __init__(self, upscale: bool, attention: bool, num_groups: int, 
                 dropout_prob: float, num_heads: int, C: int):
        super().__init__()
        self.ResBlock1 = ResBlock(C=C, num_groups=num_groups, dropout_prob=dropout_prob)
        self.ResBlock2 = ResBlock(C=C, num_groups=num_groups, dropout_prob=dropout_prob)
        if upscale:
            self.conv = nn.ConvTranspose2d(C, C//2, kernel_size=4, stride=2, padding=1)
        else:
            self.conv = nn.Conv2d(C, C*2, kernel_size=3, stride=2, padding=1)
        if attention:
            self.attention_layer = Attention(C, num_heads=num_heads, dropout_prob=dropout_prob)

    def forward(self, x, embeddings):
        x = self.ResBlock1(x, embeddings)
        if hasattr(self, 'attention_layer'):
            x = self.attention_layer(x)
        x = self.ResBlock2(x, embeddings)
        return self.conv(x), x

class SinusoidalEmbeddings(nn.Module):
    def __init__(self, time_steps: int, embed_dim: int, device):
        super().__init__()
        position = torch.arange(time_steps).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, embed_dim, 2).float() * -(np.log(10000.0) / embed_dim))
        embeddings = torch.zeros(time_steps, embed_dim, requires_grad=False)
        embeddings[:, 0::2] = torch.sin(position * div)
        embeddings[:, 1::2] = torch.cos(position * div)
        self.embeddings = embeddings.to(device)

    def forward(self, t):
        embeds = self.embeddings[t]
        return embeds[:, :, None, None]

class UNET(nn.Module):
    def __init__(self, Channels=[64, 128, 256, 512, 512, 384],
                 Attentions=[False, True, False, False, False, True],
                 Upscales=[False, False, False, True, True, True],
                 num_groups=32, dropout_prob=0.0, num_heads=8,
                 input_channels=3, output_channels=3, device='cuda', time_steps=1000):
        super().__init__()
        self.num_layers = len(Channels)
        self.shallow_conv = nn.Conv2d(input_channels, Channels[0], kernel_size=3, padding=1)
        out_channels = (Channels[-1] // 2) + Channels[0]
        self.late_conv = nn.Conv2d(out_channels, out_channels // 2, kernel_size=3, padding=1)
        self.output_conv = nn.Conv2d(out_channels // 2, output_channels, kernel_size=1)
        self.relu = nn.ReLU(inplace=True)
        self.embeddings = SinusoidalEmbeddings(time_steps=time_steps, embed_dim=max(Channels), device=device)

        for i in range(self.num_layers):
            layer = UnetLayer(
                upscale=Upscales[i], attention=Attentions[i],
                num_groups=num_groups, dropout_prob=dropout_prob,
                C=Channels[i], num_heads=num_heads
            )
            setattr(self, f'Layer{i+1}', layer)

    def forward(self, x, t):
        x = self.shallow_conv(x)
        residuals = []
        embeddings = self.embeddings(t)

        for i in range(self.num_layers // 2):
            layer = getattr(self, f'Layer{i+1}')
            x, r = layer(x, embeddings)
            residuals.append(r)

        for i in range(self.num_layers // 2, self.num_layers):
            layer = getattr(self, f'Layer{i+1}')
            x, _ = layer(x, embeddings)
            residual_idx = self.num_layers - i - 1
            if residual_idx < len(residuals):
                x = torch.cat((x, residuals[residual_idx]), dim=1)

        x = self.relu(self.late_conv(x))
        return self.output_conv(x)

In [9]:
class DDPM_Scheduler(nn.Module):
    def __init__(self, num_time_steps=1000):
        super().__init__()
        self.beta = torch.linspace(1e-4, 0.02, num_time_steps, requires_grad=False)
        alpha = 1 - self.beta
        self.alpha = torch.cumprod(alpha, dim=0).requires_grad_(False)

    def forward(self, t):
        return self.beta[t], self.alpha[t]

In [10]:
class DPOTripletDataset(Dataset):
    def __init__(self, triplets):
        self.triplets = triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        real, plausible, implausible = self.triplets[idx]
        return real, plausible, implausible

In [11]:
def ssim_comparision(
    model_old, model_new, real_noisy_image, 
    plausible_noisy_image, implausible_noisy_image, 
    t, beta, kl_weight=100.0):
    
    real_noisy_image = real_noisy_image.to(device)
    plausible_noisy_image = plausible_noisy_image.to(device)
    implausible_noisy_image = implausible_noisy_image.to(device)

    with torch.no_grad():
        gen_noisy_image_old = model_old(real_noisy_image, t)
    gen_noisy_image_new = model_new(real_noisy_image, t)

    ssim_plausible_new = torch.clamp(ssim(gen_noisy_image_new, plausible_noisy_image, data_range=1.0, size_average=True), min=1e-6, max=1.0)
    ssim_plausible_old = torch.clamp(ssim(gen_noisy_image_old, plausible_noisy_image, data_range=1.0, size_average=True), min=1e-6, max=1.0)
    ssim_implausible_new = torch.clamp(ssim(gen_noisy_image_new, implausible_noisy_image, data_range=1.0, size_average=True), min=1e-6, max=1.0)
    ssim_implausible_old = torch.clamp(ssim(gen_noisy_image_old, implausible_noisy_image, data_range=1.0, size_average=True), min=1e-6, max=1.0)

    eps = 1e-8  
    r_plus_variant = beta * torch.log((ssim_plausible_new + eps) / (ssim_plausible_old + eps))
    r_minus_variant = beta * torch.log((ssim_implausible_new + eps) / (ssim_implausible_old + eps))

    preference_score = r_plus_variant - r_minus_variant
    preference_prob = torch.sigmoid(preference_score)

    loss = -torch.log(preference_prob + eps).mean()
    kl_penalty = torch.mean((gen_noisy_image_new - gen_noisy_image_old) ** 2)

    return loss + kl_weight * kl_penalty

In [12]:
def retraining_ddpm_bone_marrow(
    checkpoint_path_new='/home/pratyushprasad/DPO_PROJECT/2. vanilla_ddpm_model/bone_marrow/saved_model/ddpm_bone_marrow_first_model.pt',
    checkpoint_path_old='/home/pratyushprasad/DPO_PROJECT/2. vanilla_ddpm_model/bone_marrow/saved_model/ddpm_bone_marrow_first_model.pt',  
    num_time_steps=1000, ema_decay=0.9999,
    train_dataloader=None, val_dataloader=None,
    save_path='/home/pratyushprasad/DPO_PROJECT/6. dpo/bone_marrow/saved_models/ddpm_bone_marrow_dpo_model_050_ssim.pt',
    lr=1e-7, early_stopping_patience=10, lr_scheduler_patience=3, lr_factor=0.5, epochs=100,
    warmup_epochs=5):
    
    scheduler_new = DDPM_Scheduler(num_time_steps=num_time_steps)
    model_new = UNET(input_channels=3, output_channels=3, device='cuda').cuda()
    optimizer_new = optim.Adam(model_new.parameters(), lr=lr)
    ema_new = ModelEmaV3(model_new, decay=ema_decay)
    checkpoint_new = torch.load(checkpoint_path_new)
    model_new.load_state_dict(checkpoint_new['weights'])
    ema_new.load_state_dict(checkpoint_new['ema'])
    optimizer_new.load_state_dict(checkpoint_new['optimizer'])
    for param_group in optimizer_new.param_groups:
        param_group['lr'] = lr
    scheduler_new.alpha = scheduler_new.alpha.cuda()
    
    scheduler_old = DDPM_Scheduler(num_time_steps=num_time_steps)
    model_old = UNET(input_channels=3, output_channels=3, device='cuda').cuda()
    checkpoint_old = torch.load(checkpoint_path_old)
    model_old.load_state_dict(checkpoint_old['weights'])
    scheduler_old.alpha = scheduler_old.alpha.cuda()
    model_old.eval()
    
    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_new, mode='min', factor=lr_factor, patience=lr_scheduler_patience)
    
    best_val_loss = float('inf')
    early_stopping_counter = 0
    print("Starting DPO training with warmup...")
    print(f"Warmup phase: {warmup_epochs} epochs")
    print(f"Main training: {epochs} epochs")
    print(f"Target learning rate: {lr:.10f}")
    
    warmup_start_lr = lr * 0.1
    total_epochs = warmup_epochs + epochs
    
    for epoch in range(total_epochs):
        is_warmup = epoch < warmup_epochs
        if is_warmup:
            warmup_lr = warmup_start_lr + (lr - warmup_start_lr) * (epoch / warmup_epochs)
            for param_group in optimizer_new.param_groups:
                param_group['lr'] = warmup_lr
            phase_name = "WARMUP"
            phase_epoch = epoch + 1
            total_phase_epochs = warmup_epochs
        else:
            phase_name = "TRAINING"
            phase_epoch = epoch - warmup_epochs + 1
            total_phase_epochs = epochs
        
        train_total_loss = 0
        model_new.train()
        for batch_idx, (real_data, plausible_data, implausible_data) in enumerate(train_dataloader):
            real_data = real_data.cuda()
            plausible_data = plausible_data.cuda()
            implausible_data = implausible_data.cuda()
            
            t = torch.randint(0, num_time_steps, (real_data.size(0),)).cuda()
            e = torch.randn_like(real_data, requires_grad=False)
            a = scheduler_new.alpha.cuda()[t].view(real_data.size(0), 1, 1, 1)
            
            real_data_noised = (torch.sqrt(a) * real_data) + (torch.sqrt(1 - a) * e)
            plausible_data_noised = (torch.sqrt(a) * plausible_data) + (torch.sqrt(1 - a) * e)
            implausible_data_noised = (torch.sqrt(a) * implausible_data) + (torch.sqrt(1 - a) * e)
            
            loss = ssim_comparision(model_old, model_new, real_data_noised, plausible_data_noised, implausible_data_noised, t, beta=10)
            
            if loss is None or not isinstance(loss, torch.Tensor):
                print(f"WARNING: Invalid loss in training at epoch {epoch}, batch {batch_idx}")
                continue
                
            optimizer_new.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_new.parameters(), max_norm=1.0)
            optimizer_new.step()
            ema_new.update(model_new)
            
            train_total_loss += loss.item()
            if batch_idx % 5 == 0:
                current_lr = optimizer_new.param_groups[0]['lr']
                print(f"[{phase_name}] Epoch {phase_epoch}/{total_phase_epochs} (Overall: {epoch+1}/{total_epochs}), "
                      f"Batch {batch_idx}/{len(train_dataloader)}, Loss: {loss.item():.6f}, LR: {current_lr:.10f}")
        
        avg_train_loss = train_total_loss / len(train_dataloader)
        print(f"[{phase_name}] Epoch {phase_epoch}/{total_phase_epochs} (Overall: {epoch+1}/{total_epochs}) - "
              f"Avg. Training Loss: {avg_train_loss:.6f}")
        
        val_total_loss = 0
        model_new.eval()
        
        for real_data, plausible_data, implausible_data in val_dataloader:
            real_data = real_data.cuda()
            plausible_data = plausible_data.cuda()
            implausible_data = implausible_data.cuda()
            
            t = torch.randint(0, num_time_steps, (real_data.size(0),)).cuda()
            e = torch.randn_like(real_data, requires_grad=False)
            a = scheduler_new.alpha.cuda()[t].view(real_data.size(0), 1, 1, 1)
            
            real_data_noised = (torch.sqrt(a) * real_data) + (torch.sqrt(1 - a) * e)
            plausible_data_noised = (torch.sqrt(a) * plausible_data) + (torch.sqrt(1 - a) * e)
            implausible_data_noised = (torch.sqrt(a) * implausible_data) + (torch.sqrt(1 - a) * e)
            
            val_loss = ssim_comparision(model_old, model_new, real_data_noised, plausible_data_noised, implausible_data_noised, t, beta=10)
            if val_loss is None or not isinstance(val_loss, torch.Tensor):
                print(f"WARNING: Invalid validation loss at epoch {epoch}")
                continue
                
            val_total_loss += val_loss.item()
        
        if len(val_dataloader) > 0:
            avg_val_loss = val_total_loss / len(val_dataloader)
        else:
            avg_val_loss = float('inf')
            
        print(f"[{phase_name}] Validation Loss: {avg_val_loss:.6f}")
        
        if not is_warmup:
            lr_scheduler.step(avg_val_loss)
            current_lr = optimizer_new.param_groups[0]['lr']
            
            if epoch > warmup_epochs:  # Only check after warmup
                prev_lr = getattr(lr_scheduler, '_last_lr', [lr])[0] if hasattr(lr_scheduler, '_last_lr') else lr
                if current_lr < prev_lr:
                    print(f"Learning rate reduced to {current_lr:.10e}")
            
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                early_stopping_counter = 0 
                checkpoint = {
                    'epoch': epoch,
                    'weights': model_new.state_dict(),
                    'ema': ema_new.state_dict(),
                    'optimizer': optimizer_new.state_dict(),
                    'val_loss': avg_val_loss,
                    'best_val_loss': best_val_loss
                }
                torch.save(checkpoint, save_path)
                print(f"[TRAINING] Model saved to {save_path} (validation improved: {avg_val_loss:.6f})")
            else:
                early_stopping_counter += 1
                print(f"EarlyStopping counter: {early_stopping_counter} out of {early_stopping_patience}")
                if early_stopping_counter >= early_stopping_patience:
                    print(f"Early stopping triggered after {epoch+1} epochs ({phase_epoch} training epochs)")
                    break
        else:
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                print(f"[WARMUP] Validation improved: {avg_val_loss:.6f} (not saving during warmup)")
        
        if epoch == warmup_epochs - 1:
            print(f"\n{'='*60}")
            print("WARMUP PHASE COMPLETED - Starting main DPO training phase")
            print(f"Final warmup learning rate: {optimizer_new.param_groups[0]['lr']:.6f}")
            print(f"Best validation loss from warmup: {best_val_loss:.6f}")
            print(f"{'='*60}\n")
    
    print("DPO training complete!")
    
    if os.path.exists(save_path):
        best_checkpoint = torch.load(save_path)
        model_new.load_state_dict(best_checkpoint['weights'])
        ema_new.load_state_dict(best_checkpoint['ema'])
        print(f"Loaded best model from main training phase")
    else:
        print("No model saved (training stopped during warmup)")
    
    return model_new, ema_new

In [13]:
if __name__ == "__main__":
    set_seed(42)
    train_triplets, val_triplets = create_triplet_dataset()

    train_dataset = DPOTripletDataset(train_triplets)
    val_dataset = DPOTripletDataset(val_triplets)

    print(f"Train dataset size: {len(train_dataset)}")
    print(f"Validation dataset size: {len(val_dataset)}")

    train_dataloader = DataLoader(train_dataset, batch_size=10, shuffle=True, num_workers=4)
    val_dataloader = DataLoader(val_dataset, batch_size=10, shuffle=False, num_workers=4)

    print(f"Train dataloader: {len(train_dataloader)} batches")
    print(f"Val dataloader: {len(val_dataloader)} batches")

    model_new, ema_new = retraining_ddpm_bone_marrow(
        checkpoint_path_new='/home/pratyushprasad/DPO_PROJECT/2. vanilla_ddpm_model/bone_marrow/saved_model/ddpm_bone_marrow_first_model.pt',
        checkpoint_path_old='/home/pratyushprasad/DPO_PROJECT/2. vanilla_ddpm_model/bone_marrow/saved_model/ddpm_bone_marrow_first_model.pt',
        num_time_steps=1000,
        ema_decay=0.9999,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        save_path='/home/pratyushprasad/DPO_PROJECT/6. dpo/bone_marrow/saved_models/ddpm_bone_marrow_dpo_model_050_ssim.pt',
        lr=1e-6,  
        early_stopping_patience=10,
        lr_scheduler_patience=2,
        lr_factor=0.5, 
        epochs=100,
    )

Loading images for all classes...
Processing class: PLM
  Processing first 050 image folders...
  Loaded 50 triplets for class PLM
Processing class: PEB
  Processing first 050 image folders...
  Loaded 47 triplets for class PEB
Processing class: MYB
  Processing first 050 image folders...
  Loaded 50 triplets for class MYB
Processing class: NGS
  Processing first 050 image folders...
  Loaded 50 triplets for class NGS
Processing class: BLA
  Processing first 050 image folders...
  Loaded 49 triplets for class BLA
Processing class: OTH
  Processing first 050 image folders...
  Loaded 48 triplets for class OTH
Processing class: NIF
  Processing first 050 image folders...
  Loaded 46 triplets for class NIF
Processing class: EBO
  Processing first 050 image folders...
  Loaded 50 triplets for class EBO
Processing class: PMO
  Processing first 050 image folders...
  Loaded 50 triplets for class PMO
Processing class: HAC
  Processing first 050 image folders...
  Loaded 50 triplets for class 